In [1]:
import pandas as pd
import sentence_transformers as st
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split, LeaveOneOut
import shap

In [2]:
embed = st.SentenceTransformer('all-mpnet-base-v2')

In [3]:
embed.encode(["This is a test sentence"])

array([[ 4.49861633e-03, -5.75653315e-02, -3.01529989e-02,
        -1.61244497e-02, -5.00141159e-02,  3.07815969e-02,
        -6.27592159e-03,  2.57758629e-02,  4.59072143e-02,
         2.00993791e-02,  5.28135151e-02, -1.74854752e-02,
         4.07894235e-03, -5.07867001e-02,  2.04986613e-02,
        -7.19272206e-03,  7.44992942e-02,  1.08755287e-02,
        -4.64666635e-02,  3.82890068e-02, -2.11628899e-02,
         7.09289638e-03,  4.49244166e-03, -3.46549340e-02,
        -4.58333045e-02,  7.24464655e-04, -1.27409576e-02,
        -3.60509045e-02,  1.83762182e-02, -1.23439776e-02,
         5.52061871e-02, -1.66890025e-02, -1.09744836e-02,
        -8.61221701e-02,  1.53252222e-06,  1.07464250e-02,
        -8.18828680e-03, -3.15868706e-02, -6.89344332e-02,
        -1.25576369e-03, -3.70296044e-03,  6.40715063e-02,
         5.97621594e-03,  4.63443510e-02, -3.11793517e-02,
         1.49158137e-02,  4.11028303e-02,  2.31718160e-02,
        -5.83220199e-02,  7.51837194e-02,  9.31296265e-0

In [4]:
csv = "powerpoint_data_en.csv"
df = pd.read_csv(csv)
classes = []

pairs = []
for _, row in df.iterrows():
    pairs.append((row['fixed title'],row['fixed image dir'], row['fixed body']))
    classes.append(0)
    pairs.append((row['waiting title'], row["waiting image dir"], row["waiting body"]))
    classes.append(1)
    pairs.append((row['not fixed title'], row["not fixed image dir"], row["not fixed body"]))
    classes.append(2)

titles  = [p[0] for p in pairs]
image_paths = [p[1] for p in pairs]
all_texts   = [p[2] for p in pairs]


In [5]:
text_embeddings  = embed.encode(all_texts, normalize_embeddings=True)
title_embeddings = embed.encode(titles, normalize_embeddings=True)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(text_embeddings, classes, test_size=0.2, random_state=42, stratify=classes)
model = SVC(random_state=42)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[11  4  3]
 [ 6  8  4]
 [ 4  3 11]]
              precision    recall  f1-score   support

           0       0.52      0.61      0.56        18
           1       0.53      0.44      0.48        18
           2       0.61      0.61      0.61        18

    accuracy                           0.56        54
   macro avg       0.56      0.56      0.55        54
weighted avg       0.56      0.56      0.55        54



In [7]:
X = text_embeddings  # or text_embeddings.cpu().numpy()
y = np.array(classes)  # or your label transformation

loo = LeaveOneOut()
y_true, y_pred, y_proba = [], [], []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = SVC(random_state=42, probability=True)
    model.fit(X_train, y_train)
    y_pred.append(model.predict(X_test)[0])
    y_true.append(y_test[0])
    y_proba.append(model.predict_proba(X_test)[0])

# Evaluate overall performance
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))
print("AUC:", roc_auc_score(y_true, y_proba, multi_class='ovr'))

[[51 23 16]
 [24 53 13]
 [17 11 62]]
              precision    recall  f1-score   support

           0       0.55      0.57      0.56        90
           1       0.61      0.59      0.60        90
           2       0.68      0.69      0.69        90

    accuracy                           0.61       270
   macro avg       0.61      0.61      0.61       270
weighted avg       0.61      0.61      0.61       270

AUC: 0.793724279835391


In [10]:
x_train, x_test, y_train, y_test, i_train, i_test = train_test_split(text_embeddings, classes, list(range(text_embeddings.shape[0])), test_size=0.2, random_state=42, stratify=classes)
model = SVC(random_state=42, probability=True)
model.fit(x_train, y_train)

def predict_proba(texts):
    embeddings = embed.encode(texts, convert_to_numpy=True)
    return model.predict_proba(embeddings)

#explainer = shap.KernelExplainer(predict_proba, [all_texts[x] for x in i_train])

In [11]:
# test_texts = [
#     "The movie was absolutely wonderful!",
#     "This film was terrible and boring."
# ]

# shap_values = explainer.shap_values(test_texts)